<a href="https://colab.research.google.com/github/paulsoriiiano/cmpe-259-project/blob/main/notebooks/sample_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sample RAG Notebook

In [1]:
# Get data from GitHub repo
!git clone https://github.com/paulsoriiiano/cmpe-259-project
!mv cmpe-259-project/data .
!rm -rf cmpe-259-project

Cloning into 'cmpe-259-project'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 100 (delta 39), reused 82 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 552.63 KiB | 3.37 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [2]:
# Install dependencies
!pip install langchain langchain_core==1.0.3 langchain_experimental langchain_community

INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.9/469.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 17.5 MB/

In [3]:
!pip install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [4]:
!pip install -qU langchain-ollama
!pip install -U ollama

In [5]:
!pip install jq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.1/757.1 kB 9.6 MB/s eta 0:00:00


## 1. Load `.json` data

In [6]:
from langchain_community.document_loaders import JSONLoader

In [7]:
# # Design JSON schema using jq
# jq_schema = """
# .[] | {
#   text: (
#     .["Park Name"] + " — " +
#     (.Description // "") + " " +
#     ((.Activities // []) | join(", ")) + " " +
#     ((.Amenities // []) | join(", ")) + " " +
#     ((.Facilities // []) | join(", ")) + " " +
#     (.["Extra Text"] // "")
#   ),
#   metadata: {
#     title: .["Park Name"],
#     url: .URL,
#     jurisdiction: .Jurisdiction
#   }
# }
# """

In [8]:
# Design JSON schema using jq
jq_schema = """
.[] | {
  text: (
    (.["Park Name"] | tostring) + " — " +
    "Park Hours: " + ((.["Park Hours"] // "") | tostring) + " " +
    "Contact Information: " + ((.["Contact Information"] // "") | tostring) + " " +
    "Dogs Allowed: " + ((.["Are dogs Allowed?"] // "") | tostring) + " " +
    ((.Description // "") | tostring) + " "
  ),
  metadata: {
    title: .["Park Name"],
    url: .URL,
    jurisdiction: .Jurisdiction
  }
}
"""

In [9]:
loader = JSONLoader("data/ca_state_parks.json",
                    jq_schema=jq_schema,
                    content_key="text")

In [10]:
docs = loader.load()
print(f"Loaded {len(docs)} documents.")

Loaded 283 documents.


In [14]:
# Check documents
lines = docs[1].page_content
print(lines)

Ahjumawi Lava Springs State Park — Park Hours: Sunrise to Sunset.Self-registration fees are due and payable upon entry. Contact Information: (530) 335-2777 Dogs Allowed: No Sunrise to Sunset. Self-registration fees are due and payable upon entry. Ahj"Where the waters come together...." is a translation of the California Indian word Ahjumawi, which is also the self describing word used by the band of Pit River of Indians who still inhabit the area.   Ahjumawi is a place of exceptional, even primeval, beauty. Brilliant aqua bays and tree studded islets only a few yards long dot the shoreline of Ja-She Creek, Crystal Springs, and Horr Pond. Over two thirds of the area is covered by recent (three to five thousand years) lava flows including vast areas of jagged black basalt.  The park is a wilderness area and most of it is extremely rugged lava rock. Visitors should prepare adequately for their visit. While there are over twenty miles of park trails by which to explore this beautiful geogr

## 2. Chunking the documents

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

In [18]:
doc_chunks = text_splitter.split_documents(docs)
print(f"Total number of chunks: {len(doc_chunks)}")

Total number of chunks: 1321


View sample doc chunks

In [19]:
doc_chunks[1]

Document(metadata={'source': '/content/data/ca_state_parks.json', 'seq_num': 1}, page_content='14 miles west of Laytonville on Branscomb Road. Westport, CA The weather can be unpredictable; layered clothing is recommended. Save the Redwoods League has helped permanently protect more than 150,000 acres in 37 of California State Parks redwood parks. Find out more at SaveTheRedwoods.org. Save the Redwoods League has helped permanently protect more than 150,000 acres in 37 of California State Parks redwood parks. Find out more at SaveTheRedwoods.org. Sign up to receive the latest news directly to your email. Have a question? Use the Contact Us Page.')

In [20]:
doc_chunks[10]

Document(metadata={'source': '/content/data/ca_state_parks.json', 'seq_num': 3}, page_content='Albany State Marine Reserve — Park Hours: No Hours Listed Contact Information: No phone number listed. Dogs Allowed: Yes Albany State Marine Reserve is at the northern end of McLaughlin Eastshore State Park.  From eastbound Interstate 80 / 580, take the Albany / Buchanan St exit.  Tun left and follow the western extension of Buchanan St to parking areas.  From westbound I-80, take the Albany Exit (Exit 13).  At Cleveland Av, turn left and proceed to Buchanan St, turn right.  Merge onto the main part of Buchanan St and make a legal U-turn to head west toward the park entrance. Sign up to receive the latest news directly to your email. Have a question? Use the Contact Us Page.')

## 3. Create Embeddings and Vector Store

### 3.1 Create embeddings

In [21]:
!pip install langchain_huggingface

In [24]:
# Create embedding model using pre-trained HuggingFace embedding model

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

### 3.2 Create vector database

In [25]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 63.9 MB/s eta 0:00:00


In [26]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    doc_chunks,       # documents (chunked)
    embedding_model   # embedding model
)

### 3.3 Querying the database

In [27]:
# Set up query
query = "Which parks allow dogs?"

# Perform similarity search
similar_docs = vector_store.similarity_search(query,
                                              k=5 # return top 5 chunks
                                              )

# Display query results
for i, doc in enumerate(similar_docs, 1):
  print(f"\n----Result {i}-----\n")
  print(doc.page_content)


----Result 1-----

Events & Picnic Reservations Coordinator at (818) 880-0398 or e-mail WRSHP.Events@parks.ca.gov . Dogs are permitted at Will Rogers State Historic Park.  Dogs must be on a leash of no more than six feet in length and under the direct control of their owner at all times while in the park.  Owners will be asked to remove their dog from the park if they are unable to control it.  Dogs are not allowed on the Backbone trail or in the adjoining Topanga State Park, but are allowed on the Rivas Canyon Trail leading to and from Temescal Gateway Park. Dogs are permitted at Will Rogers State Historic Park.  Dogs must be on a leash of no more than six feet in length and under the direct control of their owner at all times while in the park.  Owners will be asked to remove their dog from the park if they are unable to control it.  Dogs are not allowed on the Backbone trail or in the adjoining Topanga State Park, but are allowed on the Rivas Canyon Trail leading to and from Temesc

## 4. Loading the LLM

### 4.1. Experimenting with prompting

In [28]:
!pip install "huggingface_hub[inference]"
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 1.6 MB/s eta 0:00:00


In [29]:
from huggingface_hub import login
login(new_session=False)

In [30]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

In [33]:
import os
from openai import OpenAI

# Question-Answer (Q&A)
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token
)

chat = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct:groq",
    messages = [
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(chat.choices[0].message)

ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [35]:
# Summarization
text = """
The human brain is one of the most complex biological structures known to science.
Composed of roughly 86 billion neurons, it is responsible for every thought, memory, emotion, and action we perform.
Each neuron can form thousands of connections with other neurons, creating a vast and dynamic network that constantly changes as we learn and experience new things.
The brain consumes about 20% of the body’s energy, despite accounting for only 2% of its mass.
It is divided into several regions, each specializing in different functions: the frontal lobe manages decision-making and personality, the occipital lobe processes visual information, the temporal lobe handles language and memory, and the parietal lobe integrates sensory input.
Despite advances in neuroscience, many mysteries remain, such as the exact mechanisms behind consciousness and the full potential of neuroplasticity.
As research continues, the brain remains a symbol of both the limits of current understanding and the promise of future discovery.
"""

summary_prompt = f"Summarize this in one sentence:\n\n{text}"

messages = [
  {
      "role": "system",
      "content": "You are a helpful assistant that summarizes text."
  },
  {
      "role": "user",
      "content": summary_prompt

  }
]

chat = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct:groq",
    messages=messages,
    temperature=0.7,
    max_tokens=100
)

summary = chat.choices[0].message.content
print(summary)

The human brain is a complex and dynamic biological structure composed of 86 billion neurons, responsible for various functions, and although it has been extensively studied, many of its mechanisms, such as consciousness and neuroplasticity, remain poorly understood and are the subject of ongoing research.


In [39]:
# Instruction following (explaining for this case)
explain_prompt = f"Explain this summary in simple terms for a high school student:\n\n{summary}"
messages = [
    {"role": "system", "content": "You are a helpful assistant that provides explanations"},
    {"role": "user", "content": explain_prompt}
]

chat = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct:groq",
    messages=messages,
    temperature=0.7,
    max_tokens=512 # More detailed response
)

print(chat.choices[0].message.content)

Let's break it down:

The human brain is made up of about 86 billion tiny cells called neurons. These neurons help us think, move, and feel emotions. Even though scientists have been studying the brain for a long time, there's still a lot they don't know about how it works.

Think of it like a super complicated computer. We know what it can do, like play games and show pictures, but we don't fully understand how it does it all. Two things that are still a bit of a mystery are:

1. **Consciousness**: This is what makes us aware of our surroundings and ourselves. It's what makes you "you".
2. **Neuroplasticity**: This is the brain's ability to change and adapt throughout our lives. It's like the brain's ability to rewire itself.

Scientists are still trying to figure out how all of this works, and they're doing more research to learn more about the brain and its many secrets.


### 4.2. Using chat models (`Runnable`)

In [40]:
# Imports
from langchain_openai import ChatOpenAI
from langchain_community.llms import OpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [58]:
template = """
Where is this park located?

Question: {input}\n\n
Context:\n{context}
"""

In [77]:
# Using a LangChain chat model (runnable), not a prior response object
llm = ChatOpenAI(
    model = "meta-llama/Llama-3.3-70B-Instruct:groq",
    api_key = hf_token,
    base_url = "https://router.huggingface.co/v1",
    temperature = 0.7,
    max_tokens = 256
)

In [60]:
# Create prompt
prompt = ChatPromptTemplate.from_template(template)

In [61]:
# Create example text for the chain
sample_text = """
About Admiral William Standley State Recreation Area
Admiral William Standley State Recreation Area is at an elevation of 1,700 feet in the Coastal Range and features redwoods.
It is located near the headwaters of the south fork of the Eel River.
There are no facilities at this park.
"""

In [62]:
# Create chain
# prompt (the instructions) ->
# llm (follows instructions) ->
# output parser (format output)
chain = prompt | llm | StrOutputParser()

In [63]:
# Call the chain
chain.invoke({
    "input": sample_text,
    "context": ""
})

'The Admiral William Standley State Recreation Area is located in the Coastal Range, near the headwaters of the south fork of the Eel River, in California.'

## 5. Building the RAG pipeline

In [67]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [78]:
# Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [79]:
# Helper function to join document texts
# Context requires a string
def format_docs(docs):
  return "\n\n".join(d.page_content for d in docs)

In [80]:
# Prompts template

# System prompt
system_prompt_str = """
You are a helpful parks and outdoors experience virtual assistant.
Answer the following question based on the context provided.
"""
system_prompt = SystemMessagePromptTemplate.from_template(system_prompt_str)

# User prompt
user_prompt_str = "Context:{context}\n\nQuestion:\n{input}"
user_prompt = HumanMessagePromptTemplate.from_template(user_prompt_str)

# Combine both prompts
prompt = ChatPromptTemplate.from_messages(
    [system_prompt, user_prompt]
)

In [81]:
# Build rag chain
# input (query string) ->
# retriever (retrieve docs & format) + passthrough question ->
# prompt (what to do with the context and input) ->
# llm (process prompt) ->
# text (output)

rag_chain = (
    {
    "context": retriever | RunnableLambda(format_docs),
    "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [82]:
query = "List state parks that do not allow dogs."
response = rag_chain.invoke(query)

In [83]:
print(response)

Based on the provided context, the following state parks or areas do not allow dogs:

1. Topanga State Park (adjoining Will Rogers State Historic Park) - dogs are not allowed in this park.
2. Backbone trail in Will Rogers State Historic Park - dogs are not allowed on this specific trail.
3. Mitchell Caverns - pets, including dogs, are not allowed inside the caverns.
4. Visitor Center in Mitchell Caverns - pets, including dogs, are not allowed inside the visitor center.
5. Trails in Mitchell Caverns - pets, including dogs, are not allowed on the trails.
6. Certain state beaches, especially nesting areas of the western snowy plover - dogs are prohibited in these areas to protect the threatened shorebird.

Note that while dogs may not be allowed in these specific areas, some parks may allow leashed dogs in certain areas such as picnic areas, parking lots, or campgrounds. It's always best to check with the specific park for their dog policy.


## 6. Create Gradio interface

In [84]:
import gradio as gr

In [85]:
def rag_qa(query):
  try:
    response = rag_chain.invoke(query)
    return response
  except Exception as e:
    return f"Error: {str(e)}"

In [86]:
# Gradio UI
with gr.Blocks(title="Parks and Outdoors VA") as demo:
  gr.Markdown("## Retrieval-Augmented Q&A")
  gr.Markdown("Ask question about the loaded document. The system retrieves relevant content and uses an LLM to answer.")

  with gr.Row():
    user_input = gr.Textbox(placeholder="Ask something like 'Are there any state beaches?' ", label="Your Question")

  output = gr.Textbox(label="LLM Answer", lines=10)
  user_input.submit(fn=rag_qa, inputs=user_input, outputs=output)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f185c6c44375fd7df6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
